[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_strain.ipynb)

# Linear-Elastic Strain Solve 

A minimal walkthrough of FFTjax's strain-based Newton-CG elastic solver
(`problems.mechanics.solve_mechanics`) on a **two-phase composite**: a glass-fiber
reinforcement in an epoxy matrix, arranged in a square-packed pattern via
`generation.rve.make_square_composite_rve`, under a prescribed macroscopic strain.

We want to solve the mechanical equilibrium problem on this composite subject to its governing
PDE constraints:

$$
\nabla \cdot \sigma(\mathbf{x}) = 0, \qquad
\sigma = \mathbb{C}(\mathbf{x}):\varepsilon, \qquad
\varepsilon = \tfrac{1}{2}\big(\nabla u + \nabla u^\top\big)
$$

on a periodic domain, with the macroscopic average strain $\langle\varepsilon(\mathbf{x})\rangle_\Omega = \bar\varepsilon$
prescribed below.

The solver is the Krylov-based (CG-accelerated) Lippmann-Schwinger scheme: the periodic
Lippmann-Schwinger equation, discretized via trigonometric collocation and solved directly by
conjugate gradients against a fixed reference-medium Green's operator.

Because the two phases have a large stiffness contrast (~23x), the reference-medium correction is
nontrivial — the Newton-CG solve actually iterates, redistributing stress between the stiff fibers
and the compliant matrix. For the same reason we use Willot's `rotated` frequency scheme for the
Green's operator rather than the `standard` one: besides avoiding the 45° anisotropy bias of the
naive DFT discretization, it converges markedly better under high stiffness contrast.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import os
import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

`generation.rve.make_square_composite_rve` builds a square-packed 2-fiber RVE (GFRP): a matrix phase with
circular fiber cross-sections arranged on a square lattice, extruded along Z into a 3-D voxel grid. Here we use a 1-voxel-thick RVE, with a fiber volume fraction of 0.5, fiber radius of 5 μm, and voxel spacing of 0.2 μm.

Setting `nz=1` reduces the geometry to a 2-D-like slab (uniformly extruded along Z); combined with a macroscopic strain that has no out-of-plane (Z) components -- as prescribed below -- this gives a plane-strain solve.

In [ ]:
from generation.rve import make_square_composite_rve

phi     = 0.5      # target fiber volume fraction
r_fiber = 0.005     # fiber radius [mm]
dx      = 0.0002    # target voxel size [mm]

phase_np, n, L, phi_act = make_square_composite_rve(
    phi=phi,
    r_fiber=r_fiber,
    dx=dx,
    N_min=32,       # minimum number of voxels in x, y direction 
    nz=1,           # number of voxels in z direction (thickness) / along fiber axis
)


print("grid n :", n)
print("total voxels Nv:", int(np.prod(n)))
print("domain L [mm]:", tuple(f"{float(Li):.5g}" for Li in L))
print("fiber volume fraction (actual):", f"{phi_act:.4f}")

In [ ]:
# Vizualize the fibre cross-section in the XY plane (Z=0)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r",
          extent=(0, n[0]*dx*1000, 0, n[1]*dx*1000))
ax.set_title(f"Fiber cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [µm]")
ax.set_ylabel("y [µm]")
ax.set_aspect("equal")   # x/y are the same physical size -- keep the pixel scale equal too
plt.show()

## Materials

We define two isotropic materials — a glass fiber and an epoxy matrix — a common, high-contrast
(~23x stiffness ratio) composite.

Every material model in `materialmodels/` implements a common `ConstitutiveModel` interface: a
`.stiffness_tensor()` method that returns its 4th-order stiffness tensor $\mathbb{C}$ (so that
$\sigma = \mathbb{C}:\varepsilon$), regardless of the underlying parametrization —
`LinearElasticIsotropic` here (from $E$, $\nu$), a transversely isotropic model elsewhere. That
common return type is what lets `assemble_C_field` gather a per-voxel $\mathbb{C}(\mathbf{x})$
field from any mix of material models via the phase index, without needing to know which
parametrization each one uses.

In [ ]:
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from materialmodels.assembly import describe_materials

matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")
materials = [matrix, fiber]   # index 0 = matrix, 1 = fibre -- matches the phase labels below

phase = jnp.array(phase_np.reshape(-1))   # (nx,ny,nz) -> (Nv,), see markdown above

describe_materials(materials)

## Solving the mechanics problem

`solve_mechanics(n, L, phase, materials, eps_bar, ...)` is the one-call entry point: it assembles
$\mathbb{C}(\mathbf{x})$ from `materials`/`phase`, picks the reference medium (the phase-average
Lamé parameters $\lambda_0, \mu_0$ — a reasonable choice when neither phase dominates), builds the
Green's operator, and solves the periodic Lippmann-Schwinger equation for the strain field via
conjugate gradients. `scheme="rotated"` uses Willot's effective frequencies for the Green's
operator instead of the raw DFT ones — removes the 45° anisotropy bias and converges markedly
better at this composite's stiffness contrast.

Its `stepping` argument picks a single full-load solve (the default, `stepping="single"`) or a
load-stepped one; either way it returns `list[IncrementResult]` — one element here, at `t=1.0` —
so `results[0].solution` is the `ElasticitySolution`, whose `.eps`/`.sigma`/`.delta`/`.converged`
attributes we read below (it also carries an `.eps_bar` field, `None` here — only
`DisplacementBasedSolver`'s mixed-BC solve populates it).

For the full mathematical derivation — the Lippmann-Schwinger splitting, why the reference-medium
dependence cancels exactly, and the resulting CG system — see
[Mechanical Solvers](https://choROPeNt.github.io/FFTjax/documentation/theorie/mechanical) in the
documentation.

Only `n`/`L` go in -- `solve_mechanics` derives `dx` internally wherever it needs it (same
`(n, L)` convention as `GreenOperatorBasic`/`Willot`). Post-processing below derives its own `dx`
from `(n, L)` too, for the one thing that still wants it directly (plot extent);
`compute_displacement` and `IncrementalWriter` both now take `(n, L)` as well, not `dx`, so
there's no `xi_flat` to build and only one `dx` left to carry around, purely for plotting.

In [ ]:
from problems.mechanics import solve_mechanics

# we apply a small shear strain in the XY plane, with zero normal strains
eps_bar = jnp.array([
    [0.0, 1.0e-3, 0.0],
    [1.0e-3, 0.0, 0.0],
    [0.0,    0.0, 0.0],
])

results = solve_mechanics(
    n, L, phase, materials, eps_bar,
    scheme="rotated", toler_lin=1e-6, maxiter=1000,
)
sol = results[0].solution
eps, sigma, delta, converged = sol.eps, sol.sigma, sol.delta, sol.converged

print("converged    :", bool(converged))
print("tau_xy (avg) :", f"{float(jnp.mean(sigma[1, 0])):.3f}", "MPa")
print("G_xy (avg)   :", f"{float(jnp.mean(sigma[1, 0]))/(2*float(jnp.mean(eps[1, 0]))):.3f}", "MPa")

The Newton-CG solve takes real iterations to converge — the correction field is doing real work redistributing stress between the stiff fibres and the compliant matrix.

## Post-processing

Now we can visualize the results and also export them as a `.xdmf`/`.h5` pair for further
post-processing in ParaView or other visualization software, via FFTjax's `IncrementalWriter`
(the project-wide standard for field-data output). Every field here -- displacement, strain,
stress, phase -- is evaluated on the same voxel grid, so all of them are written voxel-centered
(`Center="Cell"`); there's no FEM-style node/cell split in this spectral scheme, so there's nothing
to gain from writing displacement at a different resolution than everything else.

In [ ]:
from post.fields import field_to_grid, von_mises, compute_displacement, to_voigt

# Post-processing
# return the fields to a 3-D grid for visualization and export
eps_grid   = field_to_grid(eps, n)
sigma_grid = field_to_grid(sigma, n)
u_grid     = compute_displacement(eps, eps_bar, n, L)
vm_grid    = von_mises(sigma_grid)

eps_voigt   = to_voigt(eps_grid).astype(np.float64)
sigma_voigt = to_voigt(sigma_grid).astype(np.float64)

display the stress and strain field in matplotlib.

In [ ]:
from utils.plotting import FieldPanel, plot_field_grid

VOIGT_LABELS = ["x", "y", "z", "xy", "xz", "yz"]
extent = [0.0, L[0], 0.0, L[1]]  # physical [mm] extent

phase_panel = FieldPanel(phase_np[:, :, 0], "Fiber phase", cmap="gray_r")
disp_panels = [
    FieldPanel(u_grid[:, :, 0, 0], r"Displacement $u_x$ [mm]", fmt="%.1e"),
    FieldPanel(u_grid[:, :, 0, 1], r"Displacement $u_y$ [mm]", fmt="%.1e"),
]

strain_row, stress_row = [], []
for i in [0, 1, 3]:  # normal-x, normal-y, shear-xy Voigt components
    is_shear = i == 3  # shear component: report engineering shear strain gamma = 2*epsilon
    strain_label = rf"$\gamma_{{{VOIGT_LABELS[i]}}}$" if is_shear else rf"$\varepsilon_{{{VOIGT_LABELS[i]}}}$"
    stress_label = rf"$\tau_{{{VOIGT_LABELS[i]}}}$" if is_shear else rf"$\sigma_{{{VOIGT_LABELS[i]}}}$"
    strain_row.append(FieldPanel(eps_voigt[:, :, 0, i], f"Strain {strain_label}", fmt="%.1e"))
    stress_row.append(FieldPanel(sigma_voigt[:, :, 0, i], f"Stress {stress_label}", fmt="%.1f"))

fig, axes = plot_field_grid([[phase_panel, *disp_panels], strain_row, stress_row], extent, figsize=(10, 9))
plt.show()

## Export the fields to XDMF/HDF5



In [ ]:
from utils.io.xdmf_writer import IncrementalWriter

output_dir = "output" if IN_COLAB else "../output/notebooks"  # Colab has no sibling "../output" like
                                                     # the local repo checkout does -- write to
                                                     # the current folder there instead
os.makedirs(output_dir, exist_ok=True)

with IncrementalWriter(f"{output_dir}/composite_rve_", grid_shape=n, grid_length=L) as w:
    w.write_increment(1, {
        "phase":        phase_np.astype(np.float64),
        "displacement": u_grid.astype(np.float64),
        "strain":       eps_voigt.astype(np.float64),
        "stress":       sigma_voigt.astype(np.float64),
        "von_mises":    vm_grid.astype(np.float64),
    }, time=1.0)

print(f"Wrote {output_dir}/composite_rve_.h5")
print(f"      {output_dir}/composite_rve_.xdmf")
print("Open the .xdmf in ParaView with the 'Xdmf3ReaderT' reader.")

## Next steps

- Sweep grid size for the composite RVE above — the [Benchmark](https://choROPeNt.github.io/FFTjax/documentation/benchmark#linear-elastic-strain-solve) page does exactly this and times it.
- [`lin-elastic_mixed-BC.ipynb`](./lin-elastic_mixed-BC.ipynb) covers the same geometry/materials
  under a free-lateral-surface uniaxial-*stress* condition instead — a more realistic mechanical
  test. It uses the displacement-based solver (`solvers.elliptic.vector.displacement_based`).